In [ ]:
# Importing all the libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import  accuracy_score,classification_report,ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:
df1 = pd.read_csv('modis_2021_India.csv')
df2 = pd.read_csv('modis_2022_India.csv')
df3 = pd.read_csv('modis_2023_India.csv')

In [ ]:
#print first 5 rows of the first dataset
df1.head()

In [ ]:
#Print last 5 rows
df1.tail()

In [ ]:
#Print the first 5 rows of the second dataset
df2.head()

In [ ]:
#Print the first 5 rows of the third dataset
df3.head()

In [ ]:
#Creating new DataFrame df that merges df1, df2, and df3 together row-wise, with the index reset, and then it shows the top 5 rows
df = pd.concat([df1, df2, df3], ignore_index=True)
df.head()

In [ ]:
#Shows the no of rows and columns
df.shape

In [ ]:
#Gives a summary of the DataFrame df
df.info()

In [ ]:
#Showing how many missing values are in each column
df.isnull().sum()

In [ ]:
#Shows all the duplicate rows are in DataFrame
df.duplicated().sum()

In [ ]:
#Shows all the column names of dataset
df.columns

In [ ]:
#Shows a quick statistical overview of your numeric data
df.describe().T

In [ ]:
#Counts how many times each unique value appears in the column named type
df.type.value_counts()

In [ ]:
#Shows all the categorical columns in the dataset
for col in df.columns:
  if df[col].dtype == 'object':
    print(f"Column: {col}")
    print(f"Unique values: {df[col].unique()}")
    print(f"Number of unique values: {df[col].nunique()}")
    print("-" * 50)

In [ ]:
#Creates a bar chart showing how many records are there for each fire type
plt.figure(figsize=(8, 6))
sns.countplot(x='type', data=df)
plt.title('Distribution of Fire Types')
plt.xlabel('Fire Type')
plt.ylabel('Count')
plt.show()

The count plot shows an imbalance in fire types, with 'MODIS' being significantly more frequent than 'VIIRS'.

In [ ]:
#creates a histogram showing how the confidence values are distributed in the dataset
plt.figure(figsize=(8, 6))
sns.histplot(df['confidence'], bins=20, kde=True)
plt.title('Distribution of Confidence')
plt.xlabel('Confidence')
plt.ylabel('Frequency')
plt.show()

The histogram shows a bimodal distribution of 'confidence', with most values clustered around low and high ends.

In [ ]:
# Box plot to visualize the distribution of confidence levels across different fire types
plt.figure(figsize=(8, 6))
sns.boxplot(x='type', y='confidence', data=df)
plt.title('Confidence by Fire Type')
plt.xlabel('Fire Type')
plt.ylabel('Confidence')
plt.show()

The box plot shows confidence distribution by fire type. Types 0 and 2 have wide ranges with high median values. Outliers, especially in MODIS, indicate unusually low or high confidence levels.

In [ ]:
# Scatter plot to visualize geographic locations of fires, colored by fire type
# Plot of 'latitude' vs 'longitude'
plt.figure(figsize=(10, 8))
sns.scatterplot(x='longitude', y='latitude', data=df, hue='type', s=10)
plt.title('Fire Locations by Type')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='Fire Type')
plt.show()

The scatter plot shows fire locations by latitude and longitude, colored by fire type. It reveals geographic clustering and highlights areas with high fire activity, useful for modeling.

In [ ]:
# Count plot for 'daynight'
plt.figure(figsize=(6, 4))
sns.countplot(x='daynight', data=df)
plt.title('Distribution of Day/Night Observations')
plt.xlabel('Day/Night')
plt.ylabel('Count')
plt.show()

The count plot shows the proportion of fire detections made during day vs. night, which helps understand potential differences in detection or fire behavior based on time of observation.

In [ ]:
# Count plot for 'Satellite'
plt.figure(figsize=(6, 4))
sns.countplot(x='satellite', data=df)
plt.title('Distribution of Satellite Observations')
plt.xlabel('Satellite')
plt.ylabel('Count')
plt.show()

This count plot displays the distribution of observations by satellite, highlighting which satellites contributed most to the data—useful since each may have different coverage or detection capabilities.

In [ ]:
# Count plot for 'version'
plt.figure(figsize=(6, 4))
sns.countplot(x='version', data=df)
plt.title('Distribution of Version')
plt.xlabel('Version')
plt.ylabel('Count')
plt.show()

In [ ]:
#Pairplot for numerical features (subset)
sns.pairplot(df[['latitude', 'longitude', 'brightness', 'confidence', 'frp', 'type']], hue='type', diag_kind='kde')
plt.suptitle('Pairplot of Numerical Features')
plt.show()

The pairplot reveals feature distributions and relationships by fire type, showing distinct patterns in geography, brightness, confidence, and FRP between MODIS and VIIRS.


In [ ]:
# Heatmap of correlations between numerical features
plt.figure(figsize=(10, 8))
correlation_matrix = df[['latitude', 'longitude', 'brightness', 'confidence', 'frp']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

* **Brightness & FRP**: Strong positive correlation → both reflect fire intensity.
* **Brightness & Confidence**: Moderate positive correlation → brighter fires = higher confidence.
* **FRP & Confidence**: Moderate positive correlation → intense fires = easier to detect.
* **Latitude/Longitude & Others**: Low correlation → location doesn't linearly affect fire intensity.
* **Overall**: Heatmap highlights key fire-related feature relationships, useful for modeling.


In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns

In [ ]:
numerical_cols

In [ ]:
numerical_cols = ['brightness', 'scan', 'track', 'acq_time','confidence', 'version', 'bright_t31', 'frp']
df[numerical_cols].hist(bins=50, figsize=(15, 10))
plt.suptitle('Histograms of Numerical Features')
plt.show()

* **Brightness / bright\_t31 / frp**: Show fire intensity levels.
* **Scan / track**: Indicate pixel size distribution.
* **Acq\_time**: Shows peak fire detection times.
* **Confidence**: Bimodal pattern of detection certainty.
* **Version**: Frequency of data versions used.
* **Type**: Shows fire type imbalance.
* **Overall**: Helps understand data range, trends, and prep needs.


In [ ]:
import statsmodels.api as sm
import scipy.stats as stats

# List of numerical features to check for distribution
numerical_features = ['brightness', 'confidence', 'frp', 'bright_t31', 'scan', 'track']

for feature in numerical_features:
    print(f"Analyzing distribution for: {feature}")

    # KDE Plot
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.kdeplot(df[feature], fill=True)
    plt.title(f'KDE Plot of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Density')

    # QQ Plot
    plt.subplot(1, 2, 2)
    stats.probplot(df[feature], dist="norm", plot=plt)
    plt.title(f'QQ Plot of {feature}')

    plt.tight_layout()
    plt.show()
    print("-" * 50)

* **Brightness**: Skewed, bimodal; QQ plot shows strong non-normality.
* **Confidence**: Bimodal; non-normal, especially in tails.
* **Frp**: Highly right-skewed; QQ plot confirms strong deviation.
* **Bright\_t31**: Slightly skewed; non-normal at extremes.
* **Scan / track**: Concentrated at low values with long tails; QQ plots show non-normality.

In [ ]:
# --- Temporal Analysis ---
# Convert 'acq_date' to datetime objects
df['acq_date'] = pd.to_datetime(df['acq_date'])
# Extract temporal features
df['year'] = df['acq_date'].dt.year
df['month'] = df['acq_date'].dt.month
df['day_of_week'] = df['acq_date'].dt.dayofweek # Monday=0, Sunday=6
df['day_of_year'] = df['acq_date'].dt.dayofyear
df['hour'] = df['acq_time'].astype(str).str[:2].astype(int) # Assuming acq_time is HHMM

In [ ]:
# Visualize fire detections by month
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='month', hue='month', palette='viridis', legend=False)
plt.title('Fire Detections by Month (2023)')
plt.xlabel('Month')
plt.ylabel('Number of Detections')
plt.xticks(ticks=range(12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun','Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.show()

In [ ]:
# Visualize fire detections by day of the week
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='day_of_week',hue='day_of_week', palette='viridis')
plt.title('Fire Detections by Day of Week (2023)')
plt.xlabel('Day of Week')
plt.ylabel('Number of Detections')
plt.xticks(ticks=range(7), labels=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.show()

In [ ]:
# Visualize outliers using box plots for key numerical features
plt.figure(figsize=(12, 8))
sns.boxplot(data=df[numerical_cols])
plt.title('Box Plots for Key Numerical Features')
plt.ylabel('Value')
plt.show()

* **Brightness, bright\_t31, frp**: Show wide range with high-value outliers → indicates occasional intense fires.
* **Scan, track**: Outliers present → some pixels much larger/smaller than typical.
* **Confidence**: Data concentrated at low/high ends with some mid-range outliers.
* **Acq\_time**: May show unusual detection times as outliers.
* **Version, type**: Numeric but categorical → box plots less informative for distribution, better for comparison across groups.


In [ ]:
def remove_outliers_iqr(df, column):
  Q1 = df[column].quantile(0.25)
  Q3 = df[column].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  df_cleaned = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)].copy()
  return df_cleaned

# Apply outlier removal to numerical columns
for col in numerical_cols:
  df = remove_outliers_iqr(df, col)

print("Shape after removing outliers:", df.shape)

In [ ]:
# Visualize box plots after outlier removal
plt.figure(figsize=(12, 8))
sns.boxplot(data=df[numerical_cols])
plt.title('Box Plots for Numerical Features After Outlier Removal')
plt.ylabel('Value')
plt.show()

* Outliers reduced for `'brightness'`, `'scan'`, `'track'`, `'bright_t31'`, `'frp'`.
* Whiskers closer; y-axis scale smaller.
* Boxes show cleaner, central data.
* `'confidence'`, `'acq_time'`, `'version'`, `'type'` still show original outliers.


In [ ]:
df.head()

In [ ]:
df.type.value_counts()

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns

In [ ]:
categorical_cols

In [ ]:
# Select categorical columns for encoding
categorical_cols_to_encode = ['daynight', 'satellite', 'instrument']
# Apply One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols_to_encode, drop_first=False)

In [ ]:
df_encoded.head(10)

In [ ]:
df_encoded.type.value_counts()

In [ ]:
!pip install folium

In [ ]:
# !pip install folium
import folium

# Create map and sample data
india_map = folium.Map(location=[22.351115, 78.667743], zoom_start=5)
sample_df = df_encoded.sample(n=min(10000, len(df_encoded)), random_state=42)

# Add markers
for _, row in sample_df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color='red',
        fill=True,
        fill_opacity=0.6,
        popup=f"FRP: {row['frp']:.2f}, Date: {row['acq_date'].strftime('%Y-%m-%d')}"
    ).add_to(india_map)

display(india_map)

**Fire Map Insights (India):**

Shows where fires occurred, with dense clusters marking hotspots.

Highlights regional patterns (e.g., forests or farms).

Clickable markers display fire intensity and date.

Based on a sample, so it gives a general distribution.

Dates offer limited time insight; better with time-series.

In [ ]:
# Normalize selected numerical features to standard scale
scaler = StandardScaler()
numerical_cols_to_scale = ['brightness', 'scan', 'track', 'confidence', 'bright_t31', 'frp']
df_encoded[numerical_cols_to_scale] = scaler.fit_transform(df_encoded[numerical_cols_to_scale])
df_encoded.head()

In [ ]:
df_encoded.info()

In [ ]:
# Heatmap of correlations between numerical features
plt.figure(figsize=(10, 8))
correlation_matrix = df_encoded[['brightness', 'scan', 'track', 'confidence', 'bright_t31', 'frp']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

In [ ]:
df_encoded.head()

In [ ]:
df_encoded.type.value_counts()

In [ ]:
# Separate features (X) and target variable (y)
# Assuming 'type' is the target variable you want to predict
# Drop temporal features if not intended for prediction task that uses 'type' as target
features = ['brightness', 'scan', 'track', 'confidence', 'bright_t31', 'frp']
target = 'type'

X = df_encoded[features]
y = df_encoded[target]

In [ ]:
X

In [ ]:
y

In [ ]:
!pip install -U imbalanced-learn
from imblearn.over_sampling import SMOTE

In [ ]:
# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE to the training data
X_resampled, y_resampled = smote.fit_resample(X, y)

# Check the distribution of the target variable after resampling
print("Distribution of target variable after SMOTE:")
print(y_resampled.value_counts())

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.25, random_state=42, stratify=y_resampled)

In [ ]:
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

In [ ]:
# Import Logistic Regression to Train from SKlearn
loreg = LogisticRegression(max_iter=200)
loreg.fit(X_train,y_train)
loreg_pred = loreg.predict(X_test)
score = accuracy_score(y_test,loreg_pred)
cr = classification_report(y_test,loreg_pred)

print("Logistic Regression")
print ("Accuracy Score value: {:.4f}".format(score))
print (cr)

In [ ]:
# Import Decision tree to Train from SKlearn
from sklearn.tree import DecisionTreeClassifier
dtc = DecisionTreeClassifier()
dtc.fit(X_train,y_train)
dtc_pred = dtc.predict(X_test)
score = accuracy_score(y_test,dtc_pred)
cr = classification_report(y_test,dtc_pred)

print("Decision Tree")
print ("Accuracy Score value: {:.4f}".format(score))
print (cr)

In [ ]:
dt_cm = ConfusionMatrixDisplay.from_estimator(dtc, X_test, y_test)

In [ ]:
# KNeighborsClassifier to Train from SKlearn
knnc = KNeighborsClassifier()
knnc.fit(X_train,y_train)
knn_pred = knnc.predict(X_test)
score = accuracy_score(y_test,knn_pred)
cr = classification_report(y_test,knn_pred)

print("KNeighbors Classifier")
print ("Accuracy Score value: {:.4f}".format(score))
print (cr)

In [ ]:
knn_cm = ConfusionMatrixDisplay.from_estimator(knnc, X_test, y_test)

In [ ]:
# RandomForestClassifier to train from SKlearn
rfc = RandomForestClassifier()
rfc.fit(X_train,y_train)
rfc_pred = rfc.predict(X_test)
score = accuracy_score(y_test,rfc_pred)
cr = classification_report(y_test,rfc_pred)

print("Random Forest")
print ("Accuracy Score value: {:.4f}".format(score))
print (cr)

In [ ]:
rf_cm = ConfusionMatrixDisplay.from_estimator(rfc, X_test, y_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import numpy as np

# Define a parameter grid with fewer combinations to search
param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

rfc = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    estimator=rfc,
    param_distributions=param_dist,
    n_iter=10,
    scoring='accuracy',
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, y_train)
best_rfc = random_search.best_estimator_
rfc_pred = best_rfc.predict(X_test)
rfc_acc = accuracy_score(y_test, rfc_pred)
rfc_report = classification_report(y_test, rfc_pred)

print("Random Forest with Hyperparameter Tuning (RandomizedSearch)")
print("Best Parameters:", random_search.best_params_)
print(f"Accuracy Score value: {rfc_acc:.4f}")
print(rfc_report)

ConfusionMatrixDisplay.from_estimator(best_rfc, X_test, y_test)


In [ ]:

!pip install xgboost

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
xgb.fit(X_train, y_train_encoded)

xgb_pred_encoded = xgb.predict(X_test)
xgb_pred = le.inverse_transform(xgb_pred_encoded)

score = accuracy_score(y_test, xgb_pred)
cr = classification_report(y_test, xgb_pred)

print("XGBoost Classifier")
print("Accuracy Score value: {:.4f}".format(score))
print(cr)

cm = confusion_matrix(y_test, xgb_pred, labels=le.classes_)
xgb_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
xgb_cm.plot()

In [ ]:
# with hyper parameter tuning
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 150]
}

xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')

grid = GridSearchCV(estimator=xgb, param_grid=param_grid, scoring='accuracy', cv=3, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train_encoded)

best_xgb = grid.best_estimator_
xgb_pred_encoded = best_xgb.predict(X_test)
xgb_pred = le.inverse_transform(xgb_pred_encoded)

score = accuracy_score(y_test, xgb_pred)
cr = classification_report(y_test, xgb_pred)

print("Best XGBoost Model")
print("Accuracy Score value: {:.4f}".format(score))
print(cr)

In [ ]:
# Collect the accuracies of each model
model_accuracies = {
    "Logistic Regression": accuracy_score(y_test, loreg_pred),
    "Decision Tree": accuracy_score(y_test, dtc_pred),
    "Random Forest": accuracy_score(y_test, rfc_pred),
    "Random Forest (Tuned)": accuracy_score(y_test, best_rfc.predict(X_test)),
    "KNeighbors Classifier": accuracy_score(y_test, knn_pred),
    "XGBoost Classifier": accuracy_score(y_test, xgb_pred),
    "Best XGBoost Model": accuracy_score(y_test, xgb_pred)
}

# Find the best model
best_model_name = max(model_accuracies, key=model_accuracies.get)
best_model_accuracy = model_accuracies[best_model_name]

print("Model Accuracies:")
for model, accuracy in model_accuracies.items():
    print(f"{model}: {accuracy:.4f}")

print(f"\nBest Model: {best_model_name} with Accuracy: {best_model_accuracy:.4f}")


In [ ]:
import joblib
# Save the best model
# Based on the previous output, let's assume Random Forest was the best model.
best_model = best_rfc

joblib.dump(best_model, 'Best_fire_detection_model.pkl')

# Save the StandardScaler instance
joblib.dump(scaler, 'scaler.pkl')

print("Best model and scaler saved successfully.")